In [1]:
import HTSeq
import numpy as np

from price2.reference_annotation import ReferenceAnnotation
from price2.data_collector import DataCollector
from price2.ribo_seq_run import ribo_seq_runs_from_bams



from matplotlib import pyplot as plt

import numba as nb
nb.set_num_threads(32)

In [2]:
gtf_file = '/projects/viro/maidhof/price2/orf_deconvolution_development/real_data/input_data/homo_sapiens_chr_15.gtf'
fasta_file = '/projects/viro/maidhof/price2/orf_deconvolution_development/real_data/input_data/homo_sapiens_chr_15.fasta'
bam_dir = '/projects/viro/maidhof/price2/orf_deconvolution_development/real_data/input_data/bams/'

workingdir = '/projects/viro/maidhof/price2/orf_deconvolution_development/real_data/workingdir'

In [3]:
ra = ReferenceAnnotation(gtf_file)

In [4]:
genome = dict((s.name, s) for s in HTSeq.FastaReader(fasta_file))

In [5]:
runs = ribo_seq_runs_from_bams(bam_dir, workingdir, ra)

In [6]:
data_collector = DataCollector(bam_dir, ra, genome, runs)

In [7]:
reads_db_path = f'{workingdir}/reads.db'
data_collector.collect_mappings(reads_db_path)

In [8]:
loci_db_path = f'{workingdir}/loci.db'
data_collector.collect_loci(loci_db_path)

In [9]:
runs_db_path = f'{workingdir}/runs.db'
data_collector.collect_runs(runs_db_path)

In [10]:
################################
### restart interpreter here ###
################################

from price2.orf_activity_estimator import ORFActivityEstimator


In [12]:
workingdir = '/projects/viro/maidhof/price2/orf_deconvolution_development/real_data/workingdir'
reads_db_path = f'{workingdir}/reads.db'
loci_db_path = f'{workingdir}/loci.db'
runs_db_path = f'{workingdir}/runs.db'

oae = ORFActivityEstimator(
    reads_db_path, 
    loci_db_path, 
    runs_db_path,
    )

In [13]:
oae.run_orf_deconvolution_parallel()

  0%|          | 0/1578 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
#loci_ids = [loc.id for loc in oae.loci.values()]
#
#db_paths = (oae.reads_db_path, oae.loci_db_path, oae.runs_db_path, oae.loci_results_db_path)
#
#from multiprocessing import Pool
#from price2.orf_activity_estimator import process_loc
#from tqdm import notebook
#
#
#with Pool(32) as pool:
#    a = pool.imap_unordered(process_loc, [((loc_id, db_paths)) for loc_id in loci_ids])
#    for bla in notebook.tqdm(a, total=len(loci_ids)):
#        pass
#


In [ ]:
import pandas as pd
for id, loc in oae.loci_dict.items():
    df = pd.DataFrame(loc)
    fig, axs = plt.subplots(len(df), 1, figsize=(5, 3*len(df)))
    if len(df)>1:
        for i, ax in enumerate(axs):
            ax.hist(df.iloc[i], bins=100, alpha=.3, range=(0,1))
            ax.title.set_text(df.iloc[i].name)
    fig.tight_layout()